# 02 — Feature Analysis

Analyzes the predictive features engineered from LOB and trade data: distributions, correlations, signal decay, and feature importance.

**Features**: Order book imbalance (OBI), weighted OBI, microprice deviation, spread/depth, trade flow, return/volatility, time controls

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.utils.paths import load_config
from src.data.load_data import load_processed
from src.analysis.signal_decay import compute_ic_table, plot_ic_decay, plot_feature_ic
from src.models.model_utils import FEATURE_GROUPS, get_feature_columns

config = load_config()
sns.set_theme(style='whitegrid', font_scale=1.1)

datasets = {}
for sym in config['assets']:
    datasets[sym] = load_processed(sym, 'features')
    print(f"{sym}: {datasets[sym].shape}")

## 1. Feature Groups Overview

In [ ]:
df = datasets['BTCUSDT']
for group_name, patterns in FEATURE_GROUPS.items():
    cols = get_feature_columns(df, group_name)
    print(f"\n{group_name} ({len(cols)} features):")
    print(f"  {cols[:8]}{'...' if len(cols) > 8 else ''}")

## 2. Feature Distributions

In [ ]:
key_features = ['obi_1', 'obi_5', 'weighted_obi', 'microprice_deviation',
                'trade_imbalance_1s', 'total_volume_1s', 'return_lag_1s', 'realized_vol_10s']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, feat in zip(axes.ravel(), key_features):
    if feat in df.columns:
        data = df[feat].dropna()
        ax.hist(data, bins=80, alpha=0.7, edgecolor='black', linewidth=0.3)
        ax.set_title(feat, fontsize=10)
        ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1, label=f'mean={data.mean():.4f}')
        ax.legend(fontsize=8)
plt.suptitle('BTCUSDT — Key Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Correlation Matrix

In [ ]:
corr_cols = key_features + ['y_return_1s', 'y_return_5s', 'y_return_10s', 'y_return_30s']
corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Feature-Target Correlation Matrix (BTCUSDT)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Signal Decay — IC across Horizons

In [ ]:
for sym in config['assets']:
    df = datasets[sym]
    ic_table = compute_ic_table(df, features=key_features,
                                 horizons=['1s', '5s', '10s', '30s'])
    print(f"\n{sym} — IC Table:")
    display(ic_table.round(4))
    
    fig = plot_ic_decay(ic_table, title=f"{sym} — Signal Decay")
    plt.show()

## 5. Feature IC Bar Plot

In [ ]:
for sym in config['assets']:
    df = datasets[sym]
    all_features = get_feature_columns(df, 'all')
    ic_table = compute_ic_table(df, features=all_features, horizons=['5s'])
    fig = plot_feature_ic(ic_table, horizon='5s', top_n=20,
                          title=f"{sym} — Top 20 Features by IC (5s)")
    plt.show()

## 6. Decile Analysis — OBI vs Future Return

In [ ]:
for sym in config['assets']:
    df = datasets[sym].dropna(subset=['obi_1', 'y_return_5s'])
    df['obi_decile'] = pd.qcut(df['obi_1'], 10, labels=False, duplicates='drop')
    
    decile_means = df.groupby('obi_decile')['y_return_5s'].mean() * 10000
    
    fig, ax = plt.subplots(figsize=(10, 5))
    decile_means.plot(kind='bar', ax=ax, color=plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(decile_means))))
    ax.set_title(f'{sym} — Mean 5s Return by OBI Decile', fontsize=14)
    ax.set_xlabel('OBI Decile (0=most ask-heavy, 9=most bid-heavy)')
    ax.set_ylabel('Mean Return (bps)')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    plt.tight_layout()
    plt.show()
    
    spread_val = decile_means.iloc[-1] - decile_means.iloc[0]
    print(f"{sym}: Top-minus-bottom decile spread = {spread_val:.2f} bps")

## 7. Feature Stability — Rolling IC

In [ ]:
from scipy.stats import spearmanr

sym = 'BTCUSDT'
df = datasets[sym].dropna(subset=['obi_1', 'y_return_5s']).copy()
df = df.set_index('timestamp')

window = 300  # 5-minute rolling windows
rolling_ic = []
for i in range(window, len(df), 60):
    chunk = df.iloc[i-window:i]
    ic, _ = spearmanr(chunk['obi_1'], chunk['y_return_5s'])
    rolling_ic.append({'time': chunk.index[-1], 'ic': ic})

ric = pd.DataFrame(rolling_ic)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ric['time'], ric['ic'], linewidth=1)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(ric['ic'].mean(), color='red', linestyle='--', label=f"mean IC={ric['ic'].mean():.3f}")
ax.fill_between(ric['time'], ric['ic'], 0, alpha=0.3,
                where=ric['ic'] > 0, color='green')
ax.fill_between(ric['time'], ric['ic'], 0, alpha=0.3,
                where=ric['ic'] < 0, color='red')
ax.set_title(f'{sym} — Rolling 5-min Spearman IC (OBI₁ vs 5s return)')
ax.set_ylabel('Spearman IC')
ax.legend()
plt.tight_layout()
plt.show()
print(f"IC > 0 in {(ric['ic'] > 0).mean()*100:.0f}% of windows")